# Visualize Weka speed data

In [ ]:
# Imports
import json

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from dataclasses import dataclass, asdict
from pathlib import Path

from weka_helpers import RatioContainer, ResultMap, Generation, convert_results_to_primitive, load_results_from_json

In [ ]:
# Input parameters

input_dir = Path("/rubin/bench/weka")

input_files: list[Path] = input_dir.glob("*.json")

In [ ]:
type ResultMapDataframe = dict[str,list[int|float|str]]

def add_data_to_rmd(rmd: ResultMapDataframe, rm: ResultMap, generation: str) -> ResultMapDataframe:
    rmd["index"].extend(list(range(1,len(rm.sizes)+1)))
    rmd["urls"].extend(list(rm.urls))
    rmd["sizes"].extend(list(rm.sizes))
    rmd["start_times"].extend(list(rm.start_times))
    rmd["end_times"].extend(list(rm.end_times))
    rmd["elapsed_times"].extend([x.total_seconds() for x in rm.elapsed_times])
    rmd["speeds_reported"].extend(list(rm.speeds_reported))
    rmd["generation"].extend([Generation(generation) for x in rm.sizes])
    rmd["marker_size"].extend([0.2 for x in rm.sizes])
    return rmd

def ratio_container_to_dataframe(ratio_container: RatioContainer) -> pd.DataFrame:
    rmd = {
        "index": [],
        "urls": [],
        "sizes": [],
        "start_times": [],
        "end_times": [],
        "elapsed_times": [],
        "speeds_reported": [],
        "generation": [],
        "marker_size": [],        
    }
    rmd=add_data_to_rmd(rmd, ratio_container.initial, generation="initial")
    if ratio_container.recheck:
        rmd=add_data_to_rmd(rmd, ratio_container.recheck, generation="recheck")
    return pd.DataFrame(data=rmd)
    
def plot_result(ratio_container: RatioContainer) -> None:
    plot_individual_speeds(ratio_container)
    

def plot_individual_speeds(ratio_container: RatioContainer) -> None:
    df = ratio_container_to_dataframe(ratio_container)
    max_speed = max(df.speeds_reported)
    labels = {
        "index": "index",
        "urls": "url",
        "sizes": "size (bytes)",
        "start_times": "start time",
        "end_times": "end time",
        "elapsed_times": "elapsed time",
        "speeds_reported": "speed (reported) (bytes/s)",
        "generation": "generation",
        "marker_size": "marker size",
    }    
    fig=px.scatter(
        df,
        x="end_times",
        y="speeds_reported",
        color="generation",
        size="marker_size",
        labels=labels,
        hover_data={
            "index": True,
            "urls": False,
            "sizes": True,
            "start_times": True,
            "end_times": True,
            "elapsed_times": True,
            "speeds_reported": True,
            "generation": False,
            "marker_size": False,
        },
        range_y=[0, 1.1 * max_speed],
    )
    title = (
        "Weka S3 Gateway speed\n"
        f" {ratio_container.display_name}"
        f"[{ratio_container.dataset_name}]\n"
        f"{ratio_container.count} records"
        f" sampled from {ratio_container.limit}"
    )
    fig.update_layout(
        title_text = title,
    )
    fig.show()

# Plot results

In [ ]:
res: list[RatioContainer] = []
for fn in input_files:
    rms=load_results_from_json(fn)
    res.extend(rms)

for ratio_container in res:
    plot_result(ratio_container)